# 퍼널 / 코호트 마스터 테이블 생성

## 설계 원칙

| 항목 | 결정 | 이유 |
|------|------|------|
| **Grain** | 주문 1건 = 1행 (order grain) | 퍼널은 주문 단위 이동을 추적; item grain은 112,650행으로 뻥튀기됨 |
| **payments 처리** | order_id 기준 집계 후 조인 | 결제 1:N → 직접 조인 시 카르테시안 곱 |
| **order_items 처리** | order_id 기준 집계 후 조인 | 다중 아이템 주문 처리 |
| **reviews 처리** | 최신 1건 선택 후 조인 | 중복 리뷰 543건 존재 |
| **geolocation 처리** | zip_prefix 기준 평균값 집계 후 조인 | 원본 100만 행 직접 조인 시 폭증 |

## 퍼널 단계 정의

```
구매(purchase) → 결제승인(approved) → 물류사인도(carrier) → 고객수령(delivered) → 리뷰작성(review)
```

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT     = Path('../../').resolve()
DATA_DIR = ROOT / 'data'
print(f'데이터 경로: {DATA_DIR}')

## 1. 원본 데이터 로딩

In [ ]:
orders       = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
customers    = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
order_items  = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments     = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
products     = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers      = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
category_tr  = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')
geolocation  = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')

# 타임스탬프 변환
ts_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in ts_cols:
    orders[col] = pd.to_datetime(orders[col])

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print('로딩 완료')
for name, df in [('orders',orders),('customers',customers),('order_items',order_items),
                 ('payments',payments),('reviews',reviews),('sellers',sellers)]:
    print(f'  {name:15s}: {df.shape}')

## 2. 각 테이블 집계 (order grain으로 통일)

In [ ]:
# --- 2-1. order_items: 주문별 집계 + shipping_limit_date 최솟값 보존 ---
products_with_cat = products.merge(category_tr, on='product_category_name', how='left')

items_enriched = order_items.merge(
    products_with_cat[['product_id','product_category_name_english',
                        'product_weight_g','product_length_cm',
                        'product_height_cm','product_width_cm']],
    on='product_id', how='left'
)

items_agg = (
    items_enriched
    .groupby('order_id')
    .agg(
        item_count=('order_item_id', 'max'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        product_category=('product_category_name_english',
                           lambda x: x.mode()[0] if x.notna().any() else np.nan),
        seller_id=('seller_id', 'first'),          # 주 판매자 (첫 번째 아이템 기준)
        shipping_limit_date=('shipping_limit_date', 'min'),  # 가장 이른 기한
        product_weight_g=('product_weight_g', 'max'),
        product_volume_cm3=('product_weight_g',    # 아이템별 부피는 products에서
                             lambda x: np.nan)     # 아래에서 재계산
    )
    .reset_index()
)

# 부피 = 가장 큰 아이템 기준
items_enriched['volume_cm3'] = (
    items_enriched['product_length_cm'] *
    items_enriched['product_height_cm'] *
    items_enriched['product_width_cm']
)
vol_agg = items_enriched.groupby('order_id')['volume_cm3'].max().reset_index()
items_agg = items_agg.drop(columns='product_volume_cm3').merge(vol_agg, on='order_id', how='left')
items_agg = items_agg.rename(columns={'volume_cm3': 'product_volume_cm3'})

print(f'items_agg shape: {items_agg.shape}')

In [ ]:
# --- 2-2. payments: 주문별 집계 ---
payments_agg = (
    payments
    .groupby('order_id')
    .agg(
        total_payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', lambda x: x.mode()[0]),   # 주 결제 수단
        payment_installments=('payment_installments', 'max'),   # 최대 할부 개월
        payment_type_nunique=('payment_type', 'nunique')        # 결제수단 종류 수
    )
    .reset_index()
)

print(f'payments_agg shape: {payments_agg.shape}')

In [ ]:
# --- 2-3. reviews: 최신 1건 선택 (중복 리뷰 제거) ---
reviews_dedup = (
    reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']]
)

print(f'reviews_dedup shape: {reviews_dedup.shape}')

In [ ]:
# --- 2-4. geolocation: zip_prefix별 대표 위경도 ---
geo_agg = (
    geolocation
    .groupby('geolocation_zip_code_prefix')
    .agg(
        geo_lat=('geolocation_lat', 'mean'),
        geo_lng=('geolocation_lng', 'mean')
    )
    .reset_index()
)

print(f'geo_agg shape: {geo_agg.shape}')

## 3. 마스터 테이블 병합 (기준: orders)

In [ ]:
# 고객 정보 (customer_unique_id 포함)
customers_slim = customers[[
    'customer_id', 'customer_unique_id',
    'customer_zip_code_prefix', 'customer_city', 'customer_state'
]]

# 판매자 정보
sellers_slim = sellers[[
    'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'
]]

# 순서대로 조인
df = (
    orders
    .merge(customers_slim,  on='customer_id',  how='left')
    .merge(items_agg,       on='order_id',     how='left')
    .merge(sellers_slim,    on='seller_id',    how='left')
    .merge(payments_agg,    on='order_id',     how='left')
    .merge(reviews_dedup,   on='order_id',     how='left')
)

# 고객 위경도
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geo_lat': 'customer_lat', 'geo_lng': 'customer_lng'
    }),
    on='customer_zip_code_prefix', how='left'
)

# 판매자 위경도
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'geo_lat': 'seller_lat', 'geo_lng': 'seller_lng'
    }),
    on='seller_zip_code_prefix', how='left'
)

print(f'병합 후 shape: {df.shape}')
print(f'원본 orders 행 수: {len(orders)}  →  변화 없음: {len(df) == len(orders)}')

## 4. 파생 변수 생성

In [ ]:
# --- 4-1. 퍼널 구간 소요 시간 ---

# 구간 1: 주문 → 결제 승인 (시간 단위)
df['time_purchase_to_approved_h'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600

# 구간 2: 결제 승인 → 물류사 인도 (판매자 처리 시간, 일 단위)
df['time_approved_to_carrier_d'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.days

# 구간 3: 물류사 인도 → 고객 수령 (택배사 배송 시간, 일 단위)
df['time_carrier_to_customer_d'] = (
    df['order_delivered_customer_date'] - df['order_delivered_carrier_date']
).dt.days

# 전체 리드타임 (주문 → 수령)
df['total_lead_time_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# 퍼널 마지막: 리뷰 작성 → 답변 소요일
df['review_response_days'] = (
    df['review_answer_timestamp'] - df['review_creation_date']
).dt.days

In [ ]:
# --- 4-2. 배송 지연 관련 ---

# 최종 배송 지연 여부 (고객 기준)
df['is_delayed'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype('Int8')

df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# 판매자 귀책 지연: shipping_limit_date 위반 여부
df['is_seller_late'] = (
    df['order_delivered_carrier_date'] > df['shipping_limit_date']
).astype('Int8')

df['seller_delay_days'] = (
    df['order_delivered_carrier_date'] - df['shipping_limit_date']
).dt.days

print('is_delayed 분포:')
print(df['is_delayed'].value_counts(dropna=False))
print('\nis_seller_late 분포 (판매자 귀책):')
print(df['is_seller_late'].value_counts(dropna=False))

In [ ]:
# --- 4-3. 날짜 역전 이상치 플래그 (HY 방식) ---

df['anomaly_carrier_before_approved'] = (
    df['order_delivered_carrier_date'] < df['order_approved_at']
).astype('Int8')

df['anomaly_customer_before_carrier'] = (
    df['order_delivered_customer_date'] < df['order_delivered_carrier_date']
).astype('Int8')

df['anomaly_flag'] = (
    (df['anomaly_carrier_before_approved'] == 1) |
    (df['anomaly_customer_before_carrier'] == 1)
).astype('Int8')

print('anomaly_flag 분포:')
print(df['anomaly_flag'].value_counts(dropna=False))

In [ ]:
# --- 4-4. 주문 시간 특성 ---

df['purchase_year']        = df['order_purchase_timestamp'].dt.year
df['purchase_month']       = df['order_purchase_timestamp'].dt.month
df['purchase_yearmonth']   = df['order_purchase_timestamp'].dt.to_period('M')
df['purchase_dayofweek']   = df['order_purchase_timestamp'].dt.dayofweek   # 0=월
df['purchase_dayofweek_name'] = df['order_purchase_timestamp'].dt.day_name()
df['purchase_hour']        = df['order_purchase_timestamp'].dt.hour

In [ ]:
# --- 4-5. 지역 분류 (SG 방식) ---

REGION_MAP = {
    'SP': '남동부', 'MG': '남동부', 'ES': '남동부', 'RJ': '남동부',
    'PR': '남부',   'SC': '남부',   'RS': '남부',
    'BA': '북동부', 'PE': '북동부', 'CE': '북동부', 'RN': '북동부',
    'PI': '북동부', 'MA': '북동부', 'SE': '북동부', 'AL': '북동부', 'PB': '북동부',
    'GO': '중서부', 'MT': '중서부', 'MS': '중서부', 'DF': '중서부',
    'AM': '북부',   'PA': '북부',   'AC': '북부',   'RO': '북부',
    'RR': '북부',   'AP': '북부',   'TO': '북부'
}

df['customer_region'] = df['customer_state'].map(REGION_MAP)
df['seller_region']   = df['seller_state'].map(REGION_MAP)

# 동일 주 배송 여부
df['is_same_state'] = (
    df['customer_state'] == df['seller_state']
).astype('Int8')

In [ ]:
# --- 4-6. 코호트 분석용 변수 ---

# 유저별 구매 순서
df['purchase_order_rank'] = (
    df.groupby('customer_unique_id')['order_purchase_timestamp']
    .rank(method='first')
    .astype(int)
)
df['is_first_purchase'] = (df['purchase_order_rank'] == 1).astype(int)

# 코호트 월 (유저의 첫 구매월)
first_purchase_month = (
    df[df['is_first_purchase'] == 1]
    [['customer_unique_id', 'purchase_yearmonth']]
    .rename(columns={'purchase_yearmonth': 'cohort_month'})
)
df = df.merge(first_purchase_month, on='customer_unique_id', how='left')

# 코호트 경과 개월 수
df['cohort_period_months'] = (
    (df['purchase_yearmonth'] - df['cohort_month'])
    .apply(lambda x: x.n if pd.notna(x) else np.nan)
)

print('purchase_order_rank 분포 (상위5):')
print(df['purchase_order_rank'].value_counts().head())
print(f'\n재구매 주문 비율: {(df["purchase_order_rank"] > 1).mean()*100:.1f}%')

In [ ]:
# --- 4-7. 금액 파생 변수 ---

df['freight_ratio'] = (
    df['total_freight'] / (df['total_price'] + df['total_freight'])
).where(df['total_price'] + df['total_freight'] > 0, np.nan)

df['price_per_item'] = (df['total_price'] / df['item_count']).where(df['item_count'] > 0, np.nan)

df['is_free_shipping'] = (df['total_freight'] == 0).astype('Int8')

# 할부 구간 세분화
df['installment_bucket'] = pd.cut(
    df['payment_installments'],
    bins=[0, 1, 6, 24],
    labels=['일시불', '2~6회', '7회이상'],
    right=True
)

## 5. 최종 컬럼 정리 및 검증

In [ ]:
# 최종 컬럼 순서 정의
FINAL_COLS = [
    # ── 식별자
    'order_id', 'customer_unique_id', 'customer_id',

    # ── 주문 상태 / 퍼널 타임스탬프
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',

    # ── 퍼널 구간 소요 시간
    'time_purchase_to_approved_h',
    'time_approved_to_carrier_d',
    'time_carrier_to_customer_d',
    'total_lead_time_days',
    'review_response_days',

    # ── 배송 지연
    'is_delayed', 'delay_days',
    'is_seller_late', 'seller_delay_days',

    # ── 이상치 플래그
    'anomaly_flag',
    'anomaly_carrier_before_approved',
    'anomaly_customer_before_carrier',

    # ── 고객 정보
    'customer_zip_code_prefix', 'customer_city', 'customer_state',
    'customer_region', 'customer_lat', 'customer_lng',

    # ── 판매자 정보
    'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state',
    'seller_region', 'seller_lat', 'seller_lng',
    'is_same_state',

    # ── 상품
    'item_count', 'product_category',
    'product_weight_g', 'product_volume_cm3',

    # ── 금액
    'total_price', 'total_freight', 'freight_ratio',
    'price_per_item', 'is_free_shipping',

    # ── 결제
    'total_payment_value', 'payment_type', 'payment_installments',
    'payment_type_nunique', 'installment_bucket',

    # ── 리뷰
    'review_score', 'review_creation_date', 'review_answer_timestamp',

    # ── 주문 시간 특성
    'purchase_year', 'purchase_month', 'purchase_yearmonth',
    'purchase_dayofweek', 'purchase_dayofweek_name', 'purchase_hour',

    # ── 코호트
    'purchase_order_rank', 'is_first_purchase',
    'cohort_month', 'cohort_period_months',
]

df_master = df[FINAL_COLS].copy()

print(f'최종 마스터 테이블: {df_master.shape}')
print(f'컬럼 수: {len(df_master.columns)}')
df_master.head(3)

In [ ]:
# 결측치 현황
null_rate = (df_master.isnull().mean() * 100).sort_values(ascending=False)
print('=== 결측치 비율 (5% 초과 컬럼) ===')
print(null_rate[null_rate > 5].to_string())

In [ ]:
# 핵심 지표 요약
delivered = df_master[df_master['order_status'] == 'delivered']
print('=== 퍼널 분석 기준: delivered 주문 ===')
print(f'  전체 주문:           {len(df_master):>8,}')
print(f'  배송 완료:           {len(delivered):>8,} ({len(delivered)/len(df_master)*100:.1f}%)')
print(f'  배송 지연(고객기준): {delivered["is_delayed"].sum():>8,} ({delivered["is_delayed"].mean()*100:.1f}%)')
print(f'  판매자 귀책 지연:    {delivered["is_seller_late"].sum():>8,} ({delivered["is_seller_late"].mean()*100:.1f}%)')
print(f'  이상치(anomaly):     {delivered["anomaly_flag"].sum():>8,}')
print(f'  리뷰 있는 주문:      {delivered["review_score"].notna().sum():>8,}')
print(f'\n=== 코호트 분석 기준 ===')
print(f'  고유 고객 수:        {df_master["customer_unique_id"].nunique():>8,}')
print(f'  재구매 고객 수:      {(df_master.groupby("customer_unique_id")["order_id"].count() > 1).sum():>8,}')
print(f'  관찰 기간:           {df_master["purchase_yearmonth"].min()} ~ {df_master["purchase_yearmonth"].max()}')

## 6. CSV 저장

In [ ]:
# Period 타입 → 문자열 변환 후 저장
df_save = df_master.copy()
df_save['purchase_yearmonth'] = df_save['purchase_yearmonth'].astype(str)
df_save['cohort_month']       = df_save['cohort_month'].astype(str)
df_save['installment_bucket'] = df_save['installment_bucket'].astype(str)

# 전체 (퍼널 분석용 — 모든 order_status 포함)
out_all = DATA_DIR / 'funnel_master_all.csv'
df_save.to_csv(out_all, index=False)

# 배송 완료만 (코호트 분석용)
out_delivered = DATA_DIR / 'funnel_master_delivered.csv'
df_save[df_save['order_status'] == 'delivered'].to_csv(out_delivered, index=False)

print('저장 완료')
print(f'  funnel_master_all.csv       : {df_save.shape}')
print(f'  funnel_master_delivered.csv : {df_save[df_save["order_status"]=="delivered"].shape}')
print(f'\n저장 경로: {DATA_DIR}')